In [1]:
from utils import get_hugging_face_api_key
HUGGING_FACE_API_KEY= get_hugging_face_api_key()
llm_config=  {
            "model": "meta-llama/Meta-Llama-3-8B-Instruct",
            "api_key": HUGGING_FACE_API_KEY,
            "base_url": "https://router.huggingface.co/v1",
            "api_type": "openai"
            }

## The Task

In [2]:
task = '''
        Write a concise but engaging blogpost about
      TamilNadu Culture . Make sure the blogpost is
       within 100 words.
       '''

In [3]:
import autogen

writer= autogen.AssistantAgent(
    name="Writer",
    system_message="You are a writer. You write engaging and concise " 
        "blogpost (with title) on given topics. You must polish your "
        "writing based on the feedback you receive and give a refined "
        "version. Only return your final work without additional comments.",
    llm_config=llm_config,)

C:\Anaconda\envs\aiagent\Lib\site-packages\flaml\__init__.py:20: UserWarning: flaml.automl is not available. Please install flaml[automl] to enable AutoML functionalities.
  warnings.warn("flaml.automl is not available. Please install flaml[automl] to enable AutoML functionalities.")
C:\Anaconda\envs\aiagent\Lib\site-packages\autogen\oai\gemini.py:43: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  import google.generativeai as genai


In [4]:
reply= writer.generate_reply(messages=[{"content":task,"role":"user"}])

In [5]:
print(reply)

{'content': '"Unveiling the Rich Heritage of Tamil Nadu: A Cultural Odyssey"\n\nTamil Nadu, a land of vibrant culture, is a treasure trove of ancient traditions and customs. The state\'s rich cultural heritage is a reflection of its 2,000-year-old history. From the majestic temples of Thanjavur to the colorful festivals of Pongal, every aspect of Tamil culture is a testament to its resilience and creativity. The state\'s unique cuisine, with its blend of spices and flavors, is a culinary delight. The warmth and hospitality of the Tamil people make visitors feel at home. Experience the magic of Tamil Nadu\'s culture and be enchanted!', 'refusal': None, 'role': 'assistant', 'annotations': None, 'audio': None, 'function_call': None, 'tool_calls': []}


## Adding reflection 

Create a critic agent to reflect on the work of the writer agent.

In [6]:
critic= autogen.AssistantAgent(
    name="Critic",
    is_termination_msg=lambda x: x.get("content","").find("TERMINATE") >=0,
    llm_config=llm_config,
    system_message="You are a critic. You review the work of "
                "the writer and provide constructive "
                "feedback to help improve the quality of the content.",
)

In [7]:
res= critic.initiate_chat(
    recipient=writer,
    message=task,
    max_turns=2,
    summary_method="last_msg"
)

Critic (to Writer):


        Write a concise but engaging blogpost about
      TamilNadu Culture . Make sure the blogpost is
       within 100 words.
       

--------------------------------------------------------------------------------
Writer (to Critic):

"Unveiling the Rich Heritage of Tamil Nadu: A Cultural Odyssey"

Tamil Nadu, a land of vibrant culture, is a treasure trove of ancient traditions and customs. The state's rich cultural heritage is a reflection of its 2,000-year-old history. From the majestic temples of Thanjavur to the colorful festivals of Pongal, every aspect of Tamil culture is a testament to its resilience and creativity. The state's unique cuisine, with its blend of spices and flavors, is a culinary delight. The warmth and hospitality of the Tamil people make visitors feel at home. Experience the magic of Tamil Nadu's culture and be enchanted!

--------------------------------------------------------------------------------
Critic (to Writer):

**A Promisin

## Nested chat

In [8]:
SEO_reviewer= autogen.AssistantAgent(
    name= "SEO_Reviewer",
    llm_config= llm_config,
    system_message= "You are an SEO reviewer, known for "
        "your ability to optimize content for search engines, "
        "ensuring that it ranks well and attracts organic traffic. " 
        "Make sure your suggestion is concise (within 3 bullet points), "
        "concrete and to the point. "
        "Begin the review by stating your role.",
)

In [9]:
legal_reviewer = autogen.AssistantAgent(
    name="Legal_Reviewer",
    llm_config=llm_config,
    system_message="You are a legal reviewer, known for "
        "your ability to ensure that content is legally compliant "
        "and free from any potential legal issues. "
        "Make sure your suggestion is concise (within 3 bullet points), "
        "concrete and to the point. "
        "Begin the review by stating your role.",
)

In [16]:
ethics_reviewer = autogen.AssistantAgent(
    name="Ethics_Reviewer",
    llm_config=llm_config,
    system_message="You are an ethics reviewer, known for "
        "your ability to ensure that content is ethically sound "
        "and free from any potential ethical issues. " 
        "Make sure your suggestion is concise (within 3 bullet points), "
        "concrete and to the point. "
        "Begin the review by stating your role. ",
)

In [10]:
meta_reviewer = autogen.AssistantAgent(
    name="Meta_Reviewer",
    llm_config=llm_config,
    system_message="You are a meta reviewer, you aggragate and review "
    "the work of other reviewers and give a final suggestion on the content.",
)

## Orchestrate the nested chats to solve the task

In [20]:
def reflection_message(recipient, messages, sender, config):
    return f'''Review the following content. 
            \n\n {recipient.chat_messages_for_summary(sender)[-1]['content']}'''

review_chats=[
    {
        "recipient":SEO_reviewer,
        "message":reflection_message,
        "summary_method":"last_msg",
        "summary_arg":{"summary_prompt":
                       "Return review into as JSON object only:"
                       "{'reviewer:'', 'review':''}.",},
        "max_turns":1},
    {
        "recipient": legal_reviewer, "message": reflection_message, 
        "summary_method": "last_msg",
        "summary_args": {"summary_prompt" : 
                         "Return review into as JSON object only:"
                         "{'Reviewer': '', 'Review': ''}.",},
         "max_turns": 1},
    {
        "recipient": ethics_reviewer, "message": reflection_message, 
        "summary_method": "last_msg",
        "summary_args": {"summary_prompt" : 
                         "Return review into as JSON object only:"
                         "{'reviewer': '', 'review': ''}",},
        "max_turns": 1},
    {
       "recipient":meta_reviewer,
        "message":"Aggregrate feedback from all reviewers and give final suggestions on the writing.",
        "max_turns":1},
]
        

In [21]:
critic.register_nested_chats(
    review_chats,
    trigger=writer,
)

In [22]:
res= critic.initiate_chat(
    recipient=writer,
    message=task,
    max_turns=2,
    summary_method="last_msg",
)

Critic (to Writer):


        Write a concise but engaging blogpost about
      TamilNadu Culture . Make sure the blogpost is
       within 100 words.
       

--------------------------------------------------------------------------------
Writer (to Critic):

"Unveiling the Rich Heritage of Tamil Nadu: A Cultural Odyssey"

Tamil Nadu, a land of vibrant culture, is a treasure trove of ancient traditions and customs. The state's rich cultural heritage is a reflection of its 2,000-year-old history. From the majestic temples of Thanjavur to the colorful festivals of Pongal, every aspect of Tamil culture is a testament to its resilience and creativity. The state's unique cuisine, with its blend of spices and flavors, is a culinary delight. The warmth and hospitality of the Tamil people make visitors feel at home. Experience the magic of Tamil Nadu's culture and be enchanted!

--------------------------------------------------------------------------------

********************************

In [23]:
print(res.summary)

"Unveiling the Rich Heritage of Tamil Nadu: A Cultural Odyssey"

Discover the rich culture and heritage of Tamil Nadu, India, and learn about its significance in Indian culture. This article will take you on a journey through the history, traditions, and festivals of Tamil Nadu, showcasing its unique blend of spirituality, art, and literature.

**A Land of Vibrant Culture**

Tamil Nadu, a land of vibrant culture, is a treasure trove of ancient traditions and customs. The state's rich cultural heritage is a reflection of its 2,000-year-old history. From the majestic temples of Thanjavur to the colorful festivals of Pongal, every aspect of Tamil culture is a testament to its resilience and creativity.

**A Culinary Delight**

The state's unique cuisine, with its blend of spices and flavors, is a culinary delight. The warmth and hospitality of the Tamil people make visitors feel at home. Experience the magic of Tamil Nadu's culture and be enchanted!

**Conclusion**

Tamil Nadu's culture i